In [1]:
import pyspark
from pyspark.sql import SparkSession

## Question 1

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 14:52:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [46]:
spark.version

'4.1.1'

## Question 2

In [81]:
!mkdir -p /app/data/raw/yellow/2025/11 
!wget -O /app/data/raw/yellow/2025/11/yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-06 15:18:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.37, 3.163.140.18, 3.163.140.145, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.37|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘/app/data/raw/yellow/2025/11/yellow_tripdata_2025-11.parquet’

/app/data/raw/yello 100%[===================>]  67.84M  12.5MB/s    in 5.8s    

2026-03-06 15:18:51 (11.7 MB/s) - ‘/app/data/raw/yellow/2025/11/yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [82]:
df = spark.read.parquet('/app/data/raw/yellow/2025/11')
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [83]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [84]:
repartitioned_df = df.repartition(4)
repartitioned_df.write.parquet('/app/data/partitioned/yellow/2025/11/', mode='overwrite') # Save the df as parquet files (-> 24 partitions)

In [85]:
!ls -l -h /app/data/partitioned/yellow/2025/11/

total 98M
-rw-r--r-- 1 root root   0 Mar  6 15:19 _SUCCESS
-rw-r--r-- 1 root root 25M Mar  6 15:19 part-00000-738fdb45-cac7-4aa6-9f49-483324ebf52e-c000.snappy.parquet
-rw-r--r-- 1 root root 25M Mar  6 15:19 part-00001-738fdb45-cac7-4aa6-9f49-483324ebf52e-c000.snappy.parquet
-rw-r--r-- 1 root root 25M Mar  6 15:19 part-00002-738fdb45-cac7-4aa6-9f49-483324ebf52e-c000.snappy.parquet
-rw-r--r-- 1 root root 25M Mar  6 15:19 part-00003-738fdb45-cac7-4aa6-9f49-483324ebf52e-c000.snappy.parquet


## Question 3

In [87]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

trips_15 = repartitioned_df.filter(F.to_date(df.tpep_pickup_datetime) == F.lit('2025-11-15').cast(DateType()))
trips_15.count()

162604

## Question 4

In [108]:
repartitioned_df = repartitioned_df \
    .withColumn('trip_duration_hours', F.timestamp_diff('HOUR', repartitioned_df.tpep_pickup_datetime, repartitioned_df.tpep_dropoff_datetime))

repartitioned_df.agg(F.max('trip_duration_hours')).collect()[0][0]

90

## Question 5

In [111]:
!mkdir -p /app/data/lookup
!wget -O /app/data/lookup/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-06 16:07:37--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.18, 3.163.140.127, 3.163.140.145, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘/app/data/lookup/taxi_zone_lookup.csv’

/app/data/lookup/ta 100%[===================>]  12.04K  --.-KB/s    in 0.002s  

2026-03-06 16:07:37 (6.94 MB/s) - ‘/app/data/lookup/taxi_zone_lookup.csv’ saved [12331/12331]



In [114]:
zones_df = \
    spark.read \
        .option('header','true') \
        .csv('/app/data/lookup/taxi_zone_lookup.csv')

zones_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [121]:
zones_df.registerTempTable('zones')

/app/.venv/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [122]:
repartitioned_df.registerTempTable('yellow_trips_nov_2025')

In [133]:
# Get least frequent PU location zone
least_frequent_zone = spark.sql(
    """
    SELECT 
        z.zone, 
        COUNT(1) AS trip_count
    FROM yellow_trips_nov_2025 AS t
    LEFT JOIN zones AS z ON t.PULocationID = z.LocationID
    GROUP BY z.zone
    ORDER BY trip_count ASC
    LIMIT 10
    """
)
least_frequent_zone.show(truncate=False)

[Stage 99:=============================>                            (4 + 4) / 8]

+---------------------------------------------+----------+
|zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Arden Heights                                |1         |
|Port Richmond                                |3         |
|Rikers Island                                |4         |
|Rossville/Woodrow                            |4         |
|Great Kills                                  |4         |
|Green-Wood Cemetery                          |4         |
|Jamaica Bay                                  |5         |
|Westerleigh                                  |12        |
+---------------------------------------------+----------+

